# Terrestrial Analogs for **All Martian Craters**, Analyzed Individually

This notebook processes our cleaned terrestrial CSV dataset and generates an independent set of results for each crater listed in `Dataset/Craters.csv` (Jezero, Gale, Eberswalde, Holden, Saheki, and Olympus Mons Caldera).

Similarity analysis is based on Gower distances, considering topographic features in one scenario and including albedo in another.

Geographic clusters are generated using HDBSCAN from the top-N most similar terrestrial candidates identified by our similarity model. The submitted results use HDBSCAN with fixed parameters. Tuning those parameters with an NSGA-II multi-objective search is future work; a prototype is kept in section 8 and is disabled (`RUN_NSGA2 = False`).


## 1. Imports and Reproducibility

In [1]:
# Install in the environment if missing: %pip install pandas numpy scipy scikit-learn hdbscan pymoo matplotlib seaborn folium nbformat
from pathlib import Path
import warnings, json, itertools, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.metrics.pairwise import haversine_distances
from IPython.display import display
try:
    import hdbscan
except ImportError:
    hdbscan = None
try:
    import folium
except ImportError:
    folium = None
try:
    from pymoo.core.problem import ElementwiseProblem
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.operators.crossover.sbx import SBX
    from pymoo.operators.mutation.pm import PM
    from pymoo.operators.sampling.rnd import FloatRandomSampling
    from pymoo.optimize import minimize
    PYMOO_OK = True
except ImportError:
    PYMOO_OK = False
sns.set_theme(style='whitegrid')
R_EARTH = 6371.0088
SEED = 42
rng = np.random.default_rng(SEED)
print('hdbscan:', hdbscan is not None, '| pymoo:', PYMOO_OK, '| folium:', folium is not None)

hdbscan: True | pymoo: True | folium: False


## 2. Configuration: File Paths, Models, and Feature Weights

In [2]:
# Paths are relative to Code/Methodology/. Unzip Dataset/Dataset_world_clean.zip into Dataset/ first.
CRATERS_CSV = Path('../../Dataset/Craters.csv')
EARTH_CSV = Path('../../Dataset/world_integrated_clean_albedo_lst.csv')
OUT_DIR = Path('results_all_craters_v2')
OUT_DIR.mkdir(parents=True, exist_ok=True)
SELECTED_CRATERS = None  # None: ALL sites in the CSV; or ['jezero','gale']
BLOCK_ROWS = 100_000
TOP_SIZES = (20,50,100,200,500)
TOP_CLUSTER = 500
# Offset the fact that slopes and roughness are pairs of correlated variables
GROUP_WEIGHTS = {'variability':0.30, 'slopes':0.30, 'roughness':0.30, 'albedo':0.10}
TOPO_ONLY_WEIGHTS = {'variability':1/3,'slopes':1/3,'roughness':1/3}
# Ranking used for clustering: 'topography' or 'topography_albedo'
RANKING_FOR_CLUSTERS = 'topography_albedo'
# Extra comparison with LST (experimental, not a validated thermal equivalence)
ENABLE_EXPLORATORY_LST = True
NORMALIZATION = 'robust_clip'  # robust_clip or full_range
Q_LOW, Q_HIGH = .01,.99
MIN_CLUSTER_SIZE, MIN_SAMPLES, CLUSTER_METHOD = 8,4,'eom'
SIL_SAMPLE, MEDOID_MAX_EXACT = 1200,800
RUN_NSGA2 = False  # Enable if pymoo and hdbscan are installed; cost multiplies per crater
POP_SIZE, N_GEN = 80, 40
RUN_SENSITIVITY = True
SEED = 42
print('Results:',OUT_DIR.resolve())

Results: results_all_craters_v2


## 3. Reading Both CSV Files and Auditing Their Schemas

In [3]:
def find_file(path):
    path=Path(path).expanduser()
    for p in [path,Path.cwd()/path.name,Path('data')/path.name,Path('/mnt/data')/path.name]:
        if p.is_file(): return p
    raise FileNotFoundError(f'{path} not found; update the path.')
CRATERS_CSV,EARTH_CSV=find_file(CRATERS_CSV),find_file(EARTH_CSV)
craters=pd.read_csv(CRATERS_CSV,low_memory=False)
earth_raw=pd.read_csv(EARTH_CSV,low_memory=False)
if 'site' not in craters: raise ValueError('Missing site column in Craters.csv')
if not {'lat','lon'}.issubset(earth_raw): raise ValueError('Missing lat/lon in the Earth CSV')
craters['site']=craters['site'].astype('string').str.strip().str.casefold()
if craters.site.isna().any() or craters.site.duplicated().any():
    raise ValueError('Each Martian site must be unique and non-null')
if SELECTED_CRATERS is not None:
    wanted={str(c).strip().casefold() for c in SELECTED_CRATERS}
    absent=wanted-set(craters.site)
    if absent: raise ValueError(f'Craters not found: {absent}')
    craters=craters.loc[craters.site.isin(wanted)].copy()
print('Mars:',craters[['site']].to_string(index=False))
print('Earth:',earth_raw.shape,'| craters:',len(craters))
print('Earth columns:',earth_raw.columns.tolist())

Mars:                site
              jezero
          eberswalde
                gale
              holden
              saheki
olympus_mons_caldera
Earth: (708793, 71) | craters: 6
Earth columns: ['continent', 'source', 'level', 'relief_scale_m', 'lat', 'lon', 'patch_lat_deg', 'patch_lon_deg', 'parent_lat', 'parent_lon', 'frac_ocean', 'frac_exact_flat', 'elev_mean_m', 'elev_std_m', 'elev_min_m', 'elev_max_m', 'relief_m', 'slope_mean', 'slope_p50', 'slope_p95', 'rough_mean', 'rough_p50', 'rough_p95', 'n_pix', 'pix_m', 'frac_urban', 'frac_cropland', 'frac_trees', 'frac_shrub', 'frac_grass', 'frac_bare', 'frac_wetland', 'frac_other', 'frac_water', 'lc_no_tile', 'area_km2', 'pop_total', 'pop_density', 'keep', 'alb_median', 'alb_p10', 'alb_p90', 'alb_iqr', 'alb_std', 'alb_n', 'frac_snow', 'lst_day_median_c', 'lst_day_p10_c', 'lst_day_p90_c', 'lst_day_iqr_c', 'lst_night_median_c', 'lst_night_p10_c', 'lst_night_p90_c', 'lst_night_iqr_c', 'lst_amp_median_c', 'lst_amp_p10_c', 'lst_amp_p90_c

## 4. Shared Terrestrial Data Cleaning

In [4]:
def parse_bool(s):
    if pd.api.types.is_bool_dtype(s): return s.astype('boolean')
    return s.astype('string').str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False}).astype('boolean')

earth=earth_raw.copy()
if 'earth_id' in earth: raise ValueError('earth_id already exists: check the CSV')
earth.insert(0,'earth_id',np.arange(len(earth),dtype=np.int64))
text_cols={'continent','source','level','reason','keep','lc_no_tile','earth_id'}
for c in earth.columns:
    if c not in text_cols: earth[c]=pd.to_numeric(earth[c],errors='coerce')
for c in ['keep','lc_no_tile']:
    if c in earth: earth[c]=parse_bool(earth[c])
issues=pd.DataFrame(index=earth.index)
issues['invalid_coord']=~(earth.lat.between(-90,90)&earth.lon.between(-180,180))
frac_cols=[c for c in earth if c.startswith('frac_')]
issues['invalid_fraction']=((earth[frac_cols]<0)|(earth[frac_cols]>1)).any(axis=1) if frac_cols else False
dup_cols=[c for c in ['level','lat','lon'] if c in earth]
issues['spatial_duplicate']=earth.duplicated(dup_cols,keep='first')
for c in ['n_pix','alb_n','lst_n']:
    issues[c+'_invalid']=(earth[c]<0).fillna(False) if c in earth else False
if 'n_pix' in earth: issues['n_pix_invalid'] |= earth.n_pix.eq(0).fillna(False)
for c in ['pix_m','area_km2']:
    issues[c+'_invalid']=(earth[c]<=0).fillna(False) if c in earth else False
issues['invalid_keep']=earth.keep.isna() if 'keep' in earth else False
issues['invalid_no_tile']=earth.lc_no_tile.isna() if 'lc_no_tile' in earth else False
hard=issues[['invalid_coord','invalid_fraction','spatial_duplicate','n_pix_invalid','pix_m_invalid','area_km2_invalid']].any(axis=1)
if 'keep' in earth: hard|=earth.keep.eq(False).fillna(False)
exclusion_log=issues.loc[hard].copy();exclusion_log.insert(0,'earth_id',earth.loc[hard,'earth_id'])
earth=earth.loc[~hard].copy().reset_index(drop=True)
for src,count in [('albedo','alb_n'),('temperature','lst_n')]:
    earth[src+'_available']=earth[count].fillna(0).gt(0) if count in earth else False
q=np.ones(len(earth))
if 'n_pix' in earth:q*=np.minimum(1,earth.n_pix.fillna(0).to_numpy(float)/100)
if 'frac_ocean' in earth:q*=1-earth.frac_ocean.fillna(0).clip(0,1).to_numpy(float)
if 'lc_no_tile' in earth:q*=np.where(earth.lc_no_tile.fillna(True).astype(bool),.8,1.)
earth['geom_quality']=np.clip(q,0,1)
if 'level' not in earth:earth['level']='single_level'
if {'parent_lat','parent_lon'}.issubset(earth):
    earth['parent_region']=earth.parent_lat.round(5).astype('string')+'|'+earth.parent_lon.round(5).astype('string')
else:earth['parent_region']=earth.level.astype('string')+'|'+earth.earth_id.astype('string')
earth.to_csv(OUT_DIR/'world_patches_clean.csv',index=False)
exclusion_log.to_csv(OUT_DIR/'excluded_records_audit.csv',index=False)
print('Original:',len(earth_raw),'| Clean:',len(earth),'| Excluded:',int(hard.sum()))
display(issues.sum().sort_values(ascending=False).to_frame('issues'))
display(earth.groupby('continent',dropna=False).size().to_frame('records'))

Original: 708793 | Clean: 708793 | Excluded: 0


,issues
invalid_coord,0
invalid_fraction,0
spatial_duplicate,0
n_pix_invalid,0
alb_n_invalid,0
lst_n_invalid,0
pix_m_invalid,0
area_km2_invalid,0
invalid_keep,0
sin_tile_invalid,0


,records
continent,
africa,386814
asia,213197
europe,16838
north_america,30354
oceania,24354
south_america,37236


## 5. Cross-Planet Feature Mapping and Analysis Scenarios

In [5]:
TOPO_MAP={
 'elev_std_m':'elev_std_m','slope_p50':'slope_p50','slope_p95':'slope_p95',
 'rough_p50':'rough_p50','rough_p95':'rough_p95'}
ALBEDO_MAP={'alb_median':'vis_albedo_p50'}
# Earth values are converted from °C to K: 273.15 is added to Earth in these scenarios.
LST_MAP_DAY={'lst_day_median_c':'ir_temp_p50_k'}
LST_MAP_NIGHT={'lst_night_median_c':'ir_temp_p50_k'}
SCENARIOS={'topography':TOPO_MAP,'topography_albedo':{**TOPO_MAP,**ALBEDO_MAP}}
if ENABLE_EXPLORATORY_LST:
    SCENARIOS['topography_albedo_lst_day_exploratory']={**TOPO_MAP,**ALBEDO_MAP,**LST_MAP_DAY}
    SCENARIOS['topography_albedo_lst_night_exploratory']={**TOPO_MAP,**ALBEDO_MAP,**LST_MAP_NIGHT}

# Shared global normalisation so that each variable has the same range for every crater.
# All Martian values of each variable are included; craters are not scaled separately.
def safe_num(value):
    return pd.to_numeric(pd.Series([value]),errors='coerce').iloc[0]
def terrestrial_values(col):
    v=pd.to_numeric(earth[col],errors='coerce')
    if col in LST_MAP_DAY or col in LST_MAP_NIGHT:return v+273.15
    return v
NORMALIZERS={}
for col,marscol in {k:v for mapping in SCENARIOS.values() for k,v in mapping.items()}.items():
    if col not in earth:continue
    vals=terrestrial_values(col)
    marsvals=pd.to_numeric(craters[marscol],errors='coerce') if marscol in craters else pd.Series([],dtype=float)
    marsvals=marsvals[np.isfinite(marsvals)]
    if NORMALIZATION=='robust_clip': a,b=vals.quantile([Q_LOW,Q_HIGH]).to_numpy(float)
    elif NORMALIZATION=='full_range':a,b=vals.min(),vals.max()
    else:raise ValueError('Invalid NORMALIZATION')
    if len(marsvals):a=min(a,marsvals.min());b=max(b,marsvals.max())
    if not np.isfinite(a) or not np.isfinite(b):continue
    NORMALIZERS[col]=(float(a),float(max(b-a,1e-12)))
print('Shared normalizers:',NORMALIZERS)

def build_reference(row,mapping):
    cols=[];ref=[];report=[]
    for col,mars_col in mapping.items():
        v=safe_num(row.get(mars_col,np.nan))
        good=col in earth and col in NORMALIZERS and np.isfinite(v)
        report.append({'earth':col,'mars':mars_col,'mars_value':v,'available':good})
        if good:cols.append(col);ref.append(float(v))
    # Never compare silently when a variable is missing: abort the scenario.
    if len(cols)!=len(mapping):return [],np.array([]),pd.DataFrame(report)
    return cols,np.array(ref,float),pd.DataFrame(report)

def group_weights(columns,scenario):
    groups={'variability':['elev_std_m'], 'slopes':['slope_p50','slope_p95'],
            'roughness':['rough_p50','rough_p95'],'albedo':['alb_median'],
            'lst':['lst_day_median_c','lst_night_median_c']}
    active={g:[f for f in fs if f in columns] for g,fs in groups.items()}
    active={g:fs for g,fs in active.items() if fs}
    # LST analysis: 60% topography, 20% albedo, 20% LST (exploratory).
    if 'lst_' in scenario: weights={'variability':.2,'slopes':.2,'roughness':.2,'albedo':.2,'lst':.2}
    elif scenario=='topography':weights=TOPO_ONLY_WEIGHTS
    else:weights=GROUP_WEIGHTS
    raw=np.array([weights.get(g,0) for g in active],float)
    if raw.sum()<=0:raise ValueError('Invalid weights')
    raw/=raw.sum();w=np.zeros(len(columns))
    for val,cols in zip(raw,active.values()):
        for col in cols:w[columns.index(col)]=val/len(cols)
    assert np.isclose(w.sum(),1)
    return w,active

# Numeric O(N*p) matrices, computed once per scenario (no N×N matrix).
SCENARIO_MATRICES={}
for scenario,mapping in SCENARIOS.items():
    if all(c in earth and c in NORMALIZERS for c in mapping):
        SCENARIO_MATRICES[scenario]=np.column_stack([terrestrial_values(c).to_numpy(float) for c in mapping])
print('Scenarios loaded:',list(SCENARIO_MATRICES))

Shared normalizers: {'elev_std_m': (0.6140936838536725, 1280.9228974463892), 'slope_p50': (0.03100640373905746, 15.302972698092997), 'slope_p95': (0.06960092227091476, 31.726046527536027), 'rough_p50': (0.08921408077884864, 54.772278754120975), 'rough_p95': (0.179084925175343, 108.16453848403164), 'alb_median': (0.1199999973177909, 0.36100000888109207), 'lst_day_median_c': (102.96761017983674, 224.302388752048), 'lst_night_median_c': (102.96761017983674, 194.88238962942836)}
Scenarios loaded: ['topography', 'topography_albedo', 'topography_albedo_lst_day_exploratory', 'topography_albedo_lst_night_exploratory']


## 6.  Gower Distance and Independent Ranking for Each Crater

In [6]:
# Conceptually reused from the original notebook; no quadratic world matrix.
def gower_vs_reference(data,ref,weights,low,span,block_rows=BLOCK_ROWS,contributions=True):
    X=np.asarray(data,float); ref=np.asarray(ref,float);weights=np.asarray(weights,float)
    if len(ref)!=X.shape[1] or len(ref)!=len(weights):raise ValueError('Incompatible Gower dimensions')
    if not np.isclose(weights.sum(),1) or (weights<0).any():raise ValueError('Invalid weights')
    if not np.isfinite(ref).all():raise ValueError('Non-finite Martian reference')
    out=np.full(len(X),np.nan);cont=np.full((len(X),X.shape[1]),np.nan,dtype=np.float32) if contributions else None
    rr=np.clip((ref-low)/span,0,1)
    for start in range(0,len(X),block_rows):
        end=min(len(X),start+block_rows)
        x=X[start:end];valid=np.isfinite(x)
        diff=np.abs(np.clip((x-low)/span,0,1)-rr)
        active=valid*weights;den=active.sum(axis=1)
        numerator=np.where(valid,diff,0)*active
        c=np.divide(numerator,den[:,None],out=np.zeros_like(numerator),where=den[:,None]>0)
        out[start:end]=np.where(den>0,c.sum(axis=1),np.nan)
        if contributions:cont[start:end]=c
    return out,cont

def rank_crater(row,scenario,export_dir):
    mapping=SCENARIOS[scenario]
    features,ref,report=build_reference(row,mapping)
    report.to_csv(export_dir/f'variables_{scenario}.csv',index=False)
    if not features:
        print(f'  SKIPPED {scenario}: variables missing in one of the CSVs')
        return None
    w,groups=group_weights(features,scenario)
    low=np.array([NORMALIZERS[f][0] for f in features]);span=np.array([NORMALIZERS[f][1] for f in features])
    X=SCENARIO_MATRICES[scenario]
    d,c=gower_vs_reference(X,ref,w,low,span)
    valid=np.isfinite(X).all(axis=1)&np.isfinite(d)
    if 'alb_median' in features and 'alb_n' in earth:valid &= earth.alb_n.fillna(0).gt(0).to_numpy()
    if any('lst_' in f for f in features) and 'lst_n' in earth:valid &= earth.lst_n.fillna(0).gt(0).to_numpy()
    rank=earth.loc[valid].copy()
    rank['gower_distance']=d[valid];rank['similarity_pct']=100*(1-rank.gower_distance)
    for j,f in enumerate(features):rank['contrib_'+f]=c[valid,j]
    rank=rank.sort_values(['gower_distance','earth_id'],kind='stable').reset_index(drop=True)
    rank.insert(1,'global_rank',np.arange(1,len(rank)+1))
    rank.to_csv(export_dir/f'ranking_{scenario}.csv',index=False)
    for k in TOP_SIZES:rank.head(k).to_csv(export_dir/f'top{k}_{scenario}.csv',index=False)
    pd.DataFrame({'variable':features,'mars_value':ref,'weight':w,'normalization_min':low,'normalization_span':span}).to_csv(export_dir/f'weights_and_reference_{scenario}.csv',index=False)
    print(f'  {scenario}: eligible {len(rank):,}; best distance {rank.gower_distance.iloc[0] if len(rank) else "n/a"}')
    return rank

# Implementation checks (not scientific results)
x=np.array([[0.,0.],[5.,5.],[10.,10.]])
d,c=gower_vs_reference(x,np.zeros(2),np.array([.5,.5]),np.zeros(2),np.ones(2)*10,2)
assert np.allclose(d,[0,.5,1]) and np.allclose(c.sum(axis=1),d)
print('Gower test passed')

Gower test passed


## 7. Geographic HDBSCAN Clustering, Metrics, and Medoids

In [7]:
def xyz_sphere(rad):
    la,ln=rad[:,0],rad[:,1]
    return R_EARTH*np.column_stack([np.cos(la)*np.cos(ln),np.cos(la)*np.sin(ln),np.sin(la)])

def geographic_clustering(rank):
    rank=rank.head(min(TOP_CLUSTER,len(rank))).copy()
    if len(rank)<max(5,MIN_CLUSTER_SIZE) or hdbscan is None:
        if hdbscan is None:print('  HDBSCAN not installed; skipping clustering')
        return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()
    if rank.level.nunique()>1:
        # separate levels; keep the most frequent level within the Top-K
        lev=rank.level.value_counts().idxmax()
        rank=rank.loc[rank.level.eq(lev)].copy()
        print('  HDBSCAN uses only level',lev)
    rad=np.radians(rank[['lat','lon']].to_numpy(float))
    labels=hdbscan.HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,min_samples=MIN_SAMPLES,
          metric='haversine',cluster_selection_method=CLUSTER_METHOD,core_dist_n_jobs=1).fit_predict(rad)
    rank['cluster_hdbscan']=labels
    keep=labels>=0;clust=np.unique(labels[keep])
    metric={'n_clusters':len(clust),'noise_pct':100*(~keep).mean(),'silhouette_haversine':np.nan,
            'davies_bouldin_ecef':np.nan,'gower_mean':rank.gower_distance.mean()}
    if len(clust)>=2 and keep.sum()>len(clust):
        ind=np.flatnonzero(keep)
        if len(ind)>SIL_SAMPLE:
            rr=np.random.default_rng(SEED)
            inds=[]
            for cc in clust:
                members=np.flatnonzero(labels==cc)
                take=min(len(members),max(2,int(SIL_SAMPLE*len(members)/len(ind))))
                inds.extend(rr.choice(members,take,replace=False))
            ind=np.array(inds,dtype=int)
        if len(np.unique(labels[ind]))>=2 and len(ind)>len(np.unique(labels[ind])):
            metric['silhouette_haversine']=silhouette_score(haversine_distances(rad[ind])*R_EARTH,labels[ind],metric='precomputed')
            # Global ECEF is an approximate chord distance, not an exact local projection.
            metric['davies_bouldin_ecef']=davies_bouldin_score(xyz_sphere(rad[ind]),labels[ind])
    summary=[];medoids=[]
    for cc in clust:
        part=rank.loc[rank.cluster_hdbscan==cc].copy()
        summary.append({'cluster':int(cc),'n':len(part),'gower_best':part.gower_distance.min(),
                        'gower_mean':part.gower_distance.mean()})
        subset=part
        if len(part)>MEDOID_MAX_EXACT:subset=part.sample(MEDOID_MAX_EXACT,random_state=SEED)
        coords=np.radians(subset[['lat','lon']].to_numpy(float))
        dg=haversine_distances(coords)*R_EARTH
        geo_avg=dg.mean(axis=1)
        # Geomorphological medoid using the ranking's active variables and the shared normalisation.
        f=SCENARIOS[RANKING_FOR_CLUSTERS]
        cols=list(f); weights,_=group_weights(cols,RANKING_FOR_CLUSTERS)
        data=np.column_stack([(pd.to_numeric(subset[c],errors='coerce') + (273.15 if c.startswith('lst_') else 0)).to_numpy(float) for c in cols])
        low=np.array([NORMALIZERS[c][0] for c in cols]);sp=np.array([NORMALIZERS[c][1] for c in cols])
        data=np.clip((data-low)/sp,0,1)
        dif=np.abs(data[:,None,:]-data[None,:,:]); morph_avg=(dif*weights).sum(axis=2).mean(axis=1)
        spatial=1-geo_avg/(geo_avg.max()+1e-12)
        optimal=.45*(1-subset.gower_distance.to_numpy(float))+.25*(1-morph_avg)+.15*subset.geom_quality.to_numpy(float)+.15*spatial
        for typ,ix in [('geographic',np.argmin(geo_avg)),('geomorphological',np.argmin(morph_avg)),('optimal_analog',np.argmax(optimal))]:
            r=subset.iloc[int(ix)]
            medoids.append({'cluster':int(cc),'representative_type':typ,'earth_id':r.earth_id,
               'lat':r.lat,'lon':r.lon,'gower_distance':r.gower_distance,
               'mean_geo_distance_km':geo_avg[int(ix)],'mean_morph_distance':morph_avg[int(ix)],
               'cluster_size':len(part)})
    return rank,pd.DataFrame([metric]),pd.DataFrame(medoids)

## 8. Future Work: NSGA-II Optimization (prototype, disabled)

This prototype is not used for the submitted results. It aims to optimize clustering metrics so that our keypoints are not redundant, avoiding recommendations of coordinates that are relatively close to one another.

Our terrestrial data come from 0.5° parent patches (about 55 km) split into ~5 km subpatches.


In [8]:
def pareto_optional(rank):
    if not RUN_NSGA2:return pd.DataFrame()
    if not PYMOO_OK or hdbscan is None:
        warnings.warn('Install pymoo and hdbscan to run NSGA-II');return pd.DataFrame()
    if len(rank)<40:return pd.DataFrame()
    cache={}
    maxk=min(len(rank),500)
    def evaluation(x):
        k=int(np.clip(round(x[0]),30,maxk));mcs=int(np.clip(round(x[1]),3,15));ms=int(np.clip(round(x[2]),1,10));method='eom' if x[3]<.5 else 'leaf'
        key=(k,mcs,ms,method)
        if key not in cache:
            part=rank.head(k)
            rad=np.radians(part[['lat','lon']].to_numpy(float))
            labs=hdbscan.HDBSCAN(min_cluster_size=mcs,min_samples=ms,metric='haversine',cluster_selection_method=method).fit_predict(rad)
            valid=labs>=0;cls=np.unique(labs[valid]);noise=1-valid.mean()
            if len(cls)>=2 and valid.sum()>len(cls):
                inds=np.where(valid)[0]
                if len(inds)>700:inds=np.random.default_rng(SEED).choice(inds,700,replace=False)
                if len(np.unique(labs[inds]))>=2:
                    sil=silhouette_score(haversine_distances(rad[inds])*R_EARTH,labs[inds],metric='precomputed')
                    db=davies_bouldin_score(xyz_sphere(rad[inds]),labs[inds])
                else:sil=-1;db=100
            else:sil=-1;db=100
            largest=max(((labs==c).sum() for c in cls),default=0)/max(1,valid.sum())
            cache[key]=(np.array([-sil,db,float(part.gower_distance.mean())]),np.array([2-len(cls),noise-.65,largest-.85]),
                        {'K':k,'min_cluster_size':mcs,'min_samples':ms,'method':method,'silhouette':sil,'davies_bouldin':db,
                         'gower_mean':part.gower_distance.mean(),'noise_pct':100*noise,'clusters':len(cls)})
        return cache[key]
    class CraterProblem(ElementwiseProblem):
        def __init__(self):super().__init__(n_var=4,n_obj=3,n_ieq_constr=3,xl=[30,3,1,0],xu=[maxk,15,10,1])
        def _evaluate(self,x,out,*args,**kwargs):
            obj,cons,_=evaluation(x);out['F']=obj;out['G']=cons
    result=minimize(CraterProblem(),NSGA2(pop_size=POP_SIZE,sampling=FloatRandomSampling(),
         crossover=SBX(prob=.9,eta=15),mutation=PM(eta=20),eliminate_duplicates=True),
         ('n_gen',N_GEN),seed=SEED,verbose=False)
    return pd.DataFrame([evaluation(x)[2] for x in np.atleast_2d(result.X)]) if result.X is not None else pd.DataFrame()

## 9. Processing Each Crater Separately and Exporting Its Files

In [9]:
import re

def safe_folder(name):
    return re.sub(r'[^a-z0-9_-]+','_',str(name).strip().lower()).strip('_')

def plots_and_maps(rank,clusters,medoids,path,name):
    fig,axes=plt.subplots(1,3,figsize=(16,4))
    axes[0].hist(rank.gower_distance,bins=50)
    axes[0].set_title(f'{name}: Gower distribution')
    top=rank.head(20).iloc[::-1]
    axes[1].barh(top.earth_id.astype(str),top.similarity_pct)
    axes[1].set_title('Top 20 (non-probabilistic similarity)')
    sample=rank if len(rank)<=30000 else rank.sample(30000,random_state=SEED)
    axes[2].scatter(sample.lon,sample.lat,c=sample.gower_distance,s=.6,cmap='viridis_r',rasterized=True)
    axes[2].set_title('Geographic distribution (sample view)')
    fig.tight_layout();fig.savefig(path/'summary.png',dpi=160);plt.close(fig)
    if not clusters.empty:
        fig,ax=plt.subplots(figsize=(10,5))
        ax.scatter(clusters.lon,clusters.lat,c=clusters.cluster_hdbscan,s=12,cmap='tab20')
        if not medoids.empty:
            pts=medoids.loc[medoids.representative_type=='optimal_analog']
            ax.scatter(pts.lon,pts.lat,marker='*',s=180,c='red',edgecolors='black')
        ax.set_title(name+' — geographic clusters and representatives')
        fig.tight_layout();fig.savefig(path/'clusters.png',dpi=160);plt.close(fig)
    if folium is not None:
        mp=folium.Map(location=[10,0],zoom_start=2,tiles='CartoDB positron')
        for r in rank.head(500).itertuples():
            folium.CircleMarker([r.lat,r.lon],radius=2,fill=True,
              tooltip=f'{name}: Gower {r.gower_distance:.4f}').add_to(mp)
        mp.save(str(path/'map_top500.html'))

def sensitivity_simple(rank,scenario,ref_row):
    if not RUN_SENSITIVITY:return pd.DataFrame()
    cols,ref,_=build_reference(ref_row,SCENARIOS[scenario]);base=rank.head(100)
    if not len(base):return pd.DataFrame()
    low=np.array([NORMALIZERS[f][0] for f in cols]);span=np.array([NORMALIZERS[f][1] for f in cols])
    X=SCENARIO_MATRICES[scenario]
    orig=set(base.earth_id)
    cases=[]
    for name,ww in [('base',group_weights(cols,scenario)[0]),('uniform_weights',np.ones(len(cols))/len(cols))]:
        dd,_=gower_vs_reference(X,ref,ww,low,span,contributions=False)
        valid=np.isfinite(X).all(axis=1)&np.isfinite(dd)
        top_ids=earth.loc[valid,['earth_id']].copy();top_ids['dist']=dd[valid]
        top_ids=top_ids.nsmallest(100,'dist')
        cases.append({'scenario':name,'top100_overlap':len(orig&set(top_ids.earth_id))/len(orig)})
    return pd.DataFrame(cases)

def run_crater(crater_row):
    name=str(crater_row['site']);path=OUT_DIR/safe_folder(name);path.mkdir(parents=True,exist_ok=True)
    print('\n===== CRATER:',name.upper(),'=====')
    generated_scenarios=[]
    rank=None
    for scenario in SCENARIOS:
        if scenario not in SCENARIO_MATRICES:continue
        result=rank_crater(crater_row,scenario,path)
        if result is not None:
            generated_scenarios.append(scenario)
            if scenario==RANKING_FOR_CLUSTERS: rank=result
            else: del result
    if rank is None:
        warnings.warn(f'No {RANKING_FOR_CLUSTERS} ranking for {name}; skipping clustering')
        return {'site':name,'folder':str(path),'scenarios':','.join(generated_scenarios),'clusters':0}
    clusters,cluster_metrics,medoids=geographic_clustering(rank)
    clusters.to_csv(path/'clusters_hdbscan.csv',index=False)
    cluster_metrics.to_csv(path/'hdbscan_metrics.csv',index=False)
    medoids.to_csv(path/'medoids.csv',index=False)
    pareto=pareto_optional(rank)
    pareto.to_csv(path/'pareto_nsga2.csv',index=False)
    sens=sensitivity_simple(rank,RANKING_FOR_CLUSTERS,crater_row)
    sens.to_csv(path/'sensitivity_analysis.csv',index=False)
    plots_and_maps(rank,clusters,medoids,path,name)
    return {'site':name,'folder':str(path),'scenarios':','.join(generated_scenarios),
            'candidates':len(rank),'min_distance':rank.gower_distance.min() if len(rank) else np.nan,
            'clusters':int(cluster_metrics.n_clusters.iloc[0]) if len(cluster_metrics) else 0}

# Craters run sequentially so that all rankings are not held in memory at once.
summary_rows=[]
for _,crater_row in craters.iterrows():
    summary_rows.append(run_crater(crater_row))
final_summary=pd.DataFrame(summary_rows)
final_summary.to_csv(OUT_DIR/'all_craters_summary.csv',index=False)
display(final_summary)
print('Final files in:',OUT_DIR.resolve())


===== CRATER: JEZERO =====
  topography: eligible 708,793; best distance 0.04111664656341636
  topography_albedo: eligible 708,793; best distance 0.04711855399901956
  topography_albedo_lst_day_exploratory: eligible 708,793; best distance 0.12146212299839951
  topography_albedo_lst_night_exploratory: eligible 708,793; best distance 0.09828074890429198

===== CRATER: EBERSWALDE =====
  topography: eligible 708,793; best distance 0.07610401109724453
  topography_albedo: eligible 708,793; best distance 0.07906089797081461
  topography_albedo_lst_day_exploratory: eligible 708,793; best distance 0.0983294182331906
  topography_albedo_lst_night_exploratory: eligible 708,793; best distance 0.06989478671754808

===== CRATER: GALE =====
  topography: eligible 708,793; best distance 0.28349305565747107
  topography_albedo: eligible 708,793; best distance 0.2836869878225004
  topography_albedo_lst_day_exploratory: eligible 708,793; best distance 0.2794344566521923
  topography_albedo_lst_night_e

,site,folder,scenarios,candidates,min_distance,clusters
0,jezero,results_all_craters_v2\jezero,"topography,topography_albedo,topography_albedo...",708793,0.047119,19
1,eberswalde,results_all_craters_v2\eberswalde,"topography,topography_albedo,topography_albedo...",708793,0.079061,17
2,gale,results_all_craters_v2\gale,"topography,topography_albedo,topography_albedo...",708793,0.283687,15
3,holden,results_all_craters_v2\holden,"topography,topography_albedo,topography_albedo...",708793,0.216939,15
4,saheki,results_all_craters_v2\saheki,"topography,topography_albedo,topography_albedo...",708793,0.188517,18
5,olympus_mons_caldera,results_all_craters_v2\olympus_mons_caldera,"topography,topography_albedo,topography_albedo...",708793,0.055083,21


Final files in: results_all_craters_v2


# Interpretation of Results

A fundamental part of this study involved adjusting the thresholds and internal parameters of our algorithms. One finding was that, although we could improve our evaluation metrics, we eventually reached a point where further improvements in those metrics did not change the resulting clusters.

This indicated that the configurations were converging toward the same clustering solution. The additional metric improvements mainly reflected better inter-cluster separability and smaller intra-cluster distances.
